In [ ]:
pip install PyMatching

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PyMatching import Matching

In [ ]:
class surface_code():
    def __init__(self):
        pass

    class planar():
        def __init__(self, dist):
            self.dist = dist
            self.dot_dist = 2 * dist - 1
            self.num_qubit_data = 2 * dist * (dist - 1) + 1
            self.num_qubit_measure = self.num_qubit_data - 1

            self.layout = None
            self.logical_line_indices = None

        # Utility: visualization
        def _draw_layout(self, title):
            dot_dist = self.dot_dist
            fig, ax = plt.subplots(figsize=(1.5*dot_dist, 1.5*dot_dist))

            for r in range(dot_dist):
                ax.plot([0, dot_dist - 1], [r, r], color='black', linewidth=1.5, zorder=1)
            for c in range(dot_dist):
                ax.plot([c, c], [0, dot_dist - 1], color='black', linewidth=1.5, zorder=1)

            ax.set_xticks(np.arange(dot_dist))
            ax.set_yticks(np.arange(dot_dist))
            ax.set_yticklabels([str(i) for i in range(dot_dist - 1, -1, -1)])
            ax.set_xlim(-0.7, dot_dist - 0.3)
            ax.set_ylim(-0.7, dot_dist - 0.3)
            ax.set_aspect('equal')
            plt.title(title, fontsize=14, pad=15)
            
            return ax

        # Layout
        def generate_dot_matrix(self, display=False):
            dot_dist = self.dot_dist
            
            r_range = np.arange(dot_dist) 
            c_range = np.arange(dot_dist)
            c_grid, r_grid = np.meshgrid(c_range, r_range)

            r_flat = r_grid.flatten()
            c_flat = c_grid.flatten()
            
            values = np.zeros(dot_dist * dot_dist, dtype=np.int8)
            
            values[r_flat % 2 == 0] = c_flat[r_flat % 2 == 0] % 2
            values[r_flat % 2 == 1] = np.where(c_flat[r_flat % 2 == 1] % 2 == 0, 2, 0)
            
            self.layout = values.reshape(dot_dist, dot_dist)
            
            if display:
                self.display(title=f"2D Surface Code Pattern ($d$={self.dist})")

        # Conversion to check matrix
        def generate_check_matrix_Z(self):
            if self.layout is None:
                self.generate_dot_matrix()
            
            r1, c1 = np.where(self.layout == 1)
            r0, c0 = np.where(self.layout == 0)
    
            check_matrix_Z = np.zeros((self.num_qubit_measure // 2, self.num_qubit_data), dtype=np.int8)

            manhattan_dist_Z = np.abs(r1[:, None] - r0) + np.abs(c1[:, None] - c0)
            check_matrix_Z[manhattan_dist_Z == 1] = 1
            
            self.logical_line_indices = np.where(c0 == 0)[0]

            return check_matrix_Z

        def display(self, error=None, syndrome=None, correct=None, title="Surface Code"):
            dot_dist = self.dot_dist
            ax = self._draw_layout(title)

            flat_layout = self.layout.flatten()
            c_grid, r_grid = np.meshgrid(np.arange(dot_dist), np.arange(dot_dist))
            x_coords = c_grid.flatten()
            y_coords = dot_dist - 1 - r_grid.flatten()

            # Qubit Masks
            mask_0 = (flat_layout == 0)
            mask_1 = (flat_layout == 1)
            mask_2 = (flat_layout == 2)

            # Basic layout
            ax.scatter(x_coords[mask_0], y_coords[mask_0], s=800, marker='o', 
                       facecolor='white', edgecolor='black', linewidth=2, zorder=3)
            ax.scatter(x_coords[mask_1], y_coords[mask_1], s=1000, marker='s', 
                       facecolor='lightgreen', linewidth=2, zorder=3)
            ax.scatter(x_coords[mask_1], y_coords[mask_1], s=200, marker='$Z$', color='darkgreen', zorder=4)
            ax.scatter(x_coords[mask_2], y_coords[mask_2], s=1000, marker='s', 
                       facecolor='khaki', linewidth=2, zorder=3)
            ax.scatter(x_coords[mask_2], y_coords[mask_2], s=200, marker='$X$', color='saddlebrown', zorder=4)

            # Error layout
            if error is not None:
                r0, c0 = np.where(self.layout == 0)
                data_x = c0
                data_y = dot_dist - 1 - r0

                # 발생한 에러 표시
                err_idx = np.where(error == 1)[0]
                if len(err_idx) > 0:
                    ax.scatter(data_x[err_idx], data_y[err_idx], s=900, marker='o',
                               facecolor='tomato', edgecolor='maroon', linewidth=2, zorder=5)
                    ax.scatter(data_x[err_idx], data_y[err_idx], s=300, marker='$X_{err}$', color='maroon', zorder=6)

                # 매칭된 복구 연산 표시
                if correct is not None:
                    corr_idx = np.where(correct == 1)[0]
                    if len(corr_idx) > 0:
                        ax.scatter(data_x[corr_idx], data_y[corr_idx], s=1500, marker='o',
                                   facecolor='none', edgecolor='deepskyblue', linestyle='--', linewidth=2.5, zorder=5)

            # 활성화된 measurement syndrome (Z) 표시
            if syndrome is not None:
                r1, c1 = np.where(self.layout == 1)
                syn_x = c1
                syn_y = dot_dist - 1 - r1
                
                syn_idx = np.where(syndrome == 1)[0]
                if len(syn_idx) > 0:
                    ax.scatter(syn_x[syn_idx], syn_y[syn_idx], s=1100, marker='s', 
                               facecolor='lightgreen', edgecolor='maroon', linewidth=2, zorder=4)
                    ax.scatter(syn_x[syn_idx], syn_y[syn_idx], s=200, marker='$Z$', color='maroon', zorder=5)

            plt.show()

        def simulate_simple(self, p_error, display=False):
            check_matrix_Z = self.generate_check_matrix_Z()

            m_Z = pymatching.Matching(check_matrix_Z)
            error_Z = rng.choice(2, self.num_qubit_data, p=[1 - p_error/3, p_error/3])
            syndrome_Z = check_matrix_Z @ error_Z % 2
            correct_Z = m_Z.decode(syndrome_Z)
            
            residual_error = (error_Z + correct_Z) % 2
            total_errors_on_cut = np.sum(residual_error[self.logical_line_indices])
            logical_error_triggered = bool(total_errors_on_cut % 2 == 1)

            if display:
                self.display(error=error_Z, syndrome=syndrome_Z, correct=correct_Z, 
                             title=f"Simulation Result (Success: {not logical_error_triggered})")

            return logical_error_triggered
            

In [ ]:
def result(d_list=[3, 7, 11, 25, 55], p_error_list=np.logspace(-4, -1, 10), num_iterations=10000):
    plt.figure(figsize=(10, 6))

    all_results = {}
    for d in d_list:
        print(f"========================================")
        print(f"[Distance d = {d}] 계산 시작")
        print(f"========================================")
        error_rates = []

        code = surface_code.check_matrix(d)

        current_p_list = p_error_list
        if d < 30:
            pass
        else:
            current_p_list = p_error_list[math.ceil(0.3 * len(p_error_list)) : math.ceil(0.7 * len(p_error_list))]
        
        for p_error in current_p_list:
            print(f" -> p_error = {p_error:.4f} 시뮬레이션 중...", flush=True)
            error_count = 0

            for _ in range(num_iterations):
                if code.simulate_rule1_Z(p_error):
                    error_count += 1
                    
            error_rate = error_count / num_iterations
            error_rates.append(error_rate)

        all_results[d] = error_rates
        plt.plot(current_p_list, error_rates, marker='o', label=f'd = {d}')

    plt.title('Numerical simulations of surface code error rates', fontsize=14, fontweight='bold')
    plt.xlabel('Physical Error Probability', fontsize=12)
    plt.ylabel('Logical X Error Probability', fontsize=12)

    plt.yscale('log')

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(title='Distance (d)', fontsize=10)
    plt.show()

In [ ]:
result(d_list=[3, 5, 7, 9, 11], p_error_list=np.linspace(0.1, 0.5, 10), num_iterations=1000)